# Directed Message Passing Neural Network

In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from lightning import pytorch as pl
from chemprop import data, featurizers

## 1. Featurize & Load Data

In [2]:
import pandas as pd

In [4]:
tox21_tasks = ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
               'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']

In [7]:
from chemprop import data

tox21_data = pd.read_csv('../data/tox21.csv')
smiles = tox21_data.loc[:, 'smiles'].values
targets = tox21_data.loc[:, tox21_tasks].values
num_workers = 0

# Convert to Chemprop's MoleculeDatapoint format
all_data = [data.MoleculeDatapoint.from_smi(smile, target) for smile, target in zip(smiles, targets)]

[14:44:35] WARNING: not removing hydrogen atom without neighbors


In [10]:
# Transform into RDkit Mol objects for structure based splits
mols = [data.mol for data in all_data]
train_indices, val_indices, test_indices = data.make_split_indices(mols, "random", (0.7, 0.2, 0.1))
train_data, val_data, test_data = data.split_data_by_indices(
    all_data, train_indices, val_indices, test_indices
)

The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)


In [13]:
from chemprop import data, featurizers

# Featurize the data
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

train_data = data.MoleculeDataset(train_data[0], featurizer)
val_data = data.MoleculeDataset(val_data[0], featurizer)
test_data = data.MoleculeDataset(test_data[0], featurizer)

# Create dataloaders
train_loader = data.build_dataloader(train_data, num_workers=num_workers)
val_loader = data.build_dataloader(val_data, num_workers=num_workers, shuffle=False)
test_loader = data.build_dataloader(test_data, num_workers=num_workers, shuffle=False)

Dropping last batch of size 1 to avoid issues with batch normalization (dataset size = 1601, batch_size = 64)


## 2. Set Metric Table

In [14]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

def compute_classification_report(y_true, y_probs, threshold=0.5):
    num_tasks = y_true.shape[1]
    metrics = {
        "task": [],
        "precision": [],
        "recall": [],
        "f1": [],
        "auc": [],
        "positives": [],
        "predicted_positives": [],
        "total_num": []
    }

    for i in range(num_tasks):
        mask = ~np.isnan(y_true[:, i])
        y_t = y_true[mask, i]
        y_p = y_probs[mask, i]
        
        y_pred = (y_p >= threshold).astype(int)

        if len(np.unique(y_t)) < 2:
            # Skip if only one class present
            metrics["task"].append(f"task_{i}")
            metrics["precision"].append(np.nan)
            metrics["recall"].append(np.nan)
            metrics["f1"].append(np.nan)
            metrics["auc"].append(np.nan)
            metrics["positives"].append(int(y_t.sum()))
            metrics["predicted_positives"].append(int(y_pred.sum()))
            metrics["total_num"].append(int(len(y_t)))
            continue


        metrics["task"].append(f"task_{i}")
        metrics["precision"].append(precision_score(y_t, y_pred, zero_division=0))
        metrics["recall"].append(recall_score(y_t, y_pred, zero_division=0))
        metrics["f1"].append(f1_score(y_t, y_pred, zero_division=0))
        metrics["auc"].append(roc_auc_score(y_t, y_p))
        metrics["positives"].append(int(y_t.sum()))
        metrics["predicted_positives"].append(int(y_pred.sum()))
        metrics["total_num"].append(int(len(y_t)))
        
    # print average metrics
    avg_precision = np.nanmean(metrics["precision"])
    avg_recall = np.nanmean(metrics["recall"])
    avg_f1 = np.nanmean(metrics["f1"])
    avg_auc = np.nanmean(metrics["auc"])
    print(f" - Precision: {avg_precision:.4f}")
    print(f" - Recall: {avg_recall:.4f}")
    print(f" - F1: {avg_f1:.4f}")
    print(f" - AUC: {avg_auc:.4f}")
        

    return pd.DataFrame(metrics)

## 3. Model

### 1 ) Initial D-MPNN

In [ ]:
import chemprop

mp = chemprop.nn.BondMessagePassing()
agg = chemprop.nn.MeanAggregation()
# print(nn.agg.AggregationRegistry)

ffn = chemprop.nn.BinaryClassificationFFN(n_tasks=len(tox21_tasks))
# print(nn.PredictorRegistry)

In [18]:
'''
Parameters
    ----------
    message_passing : MessagePassing
        the message passing block to use to calculate learned fingerprints
    agg : Aggregation
        the aggregation operation to use during molecule-level predictor
    predictor : Predictor
        the function to use to calculate the final prediction
    batch_norm : bool, default=False
        if `True`, apply batch normalization to the output of the aggregation operation
    metrics : Iterable[Metric] | None, default=None
        the metrics to use to evaluate the model during training and evaluation
    warmup_epochs : int, default=2
        the number of epochs to use for the learning rate warmup
    init_lr : int, default=1e-4
        the initial learning rate
    max_lr : float, default=1e-3
        the maximum learning rate
    final_lr : float, default=1e-4
        the final learning rate
'''

batch_norm = False
metric_list = None   # AUROC used by default
mpnn = chemprop.models.MPNN(mp, agg, ffn, batch_norm, metric_list)
mpnn

MPNN(
  (message_passing): BondMessagePassing(
    (W_i): Linear(in_features=86, out_features=300, bias=False)
    (W_h): Linear(in_features=300, out_features=300, bias=False)
    (W_o): Linear(in_features=372, out_features=300, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (tau): ReLU()
    (V_d_transform): Identity()
    (graph_transform): Identity()
  )
  (agg): MeanAggregation()
  (bn): Identity()
  (predictor): BinaryClassificationFFN(
    (ffn): MLP(
      (0): Sequential(
        (0): Linear(in_features=300, out_features=300, bias=True)
      )
      (1): Sequential(
        (0): ReLU()
        (1): Dropout(p=0.0, inplace=False)
        (2): Linear(in_features=300, out_features=12, bias=True)
      )
    )
    (criterion): BCELoss(task_weights=[[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]])
    (output_transform): Identity()
  )
  (X_d_transform): Identity()
  (metrics): ModuleList(
    (0): BinaryAUROC()
    (1): BCELoss(task_weights=[[1.0, 1.0, 1.

#### · Train

In [19]:
from lightning import pytorch as pl

In [26]:
trainer_mpnn = pl.Trainer(
    logger=False,
    enable_checkpointing=False, # Use `True` if you want to save model checkpoints. The checkpoints will be saved in the `checkpoints` folder.
    enable_progress_bar=True,
    accelerator="cpu",
    devices=1,
    max_epochs=20, # number of epochs to train for
)

trainer_mpnn.fit(mpnn, train_loader)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
Loading `train_dataloader` to estimate number of stepping batches.
/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

  | Name            | Type                    | Params | Mode 
----------------------------------------------

Epoch 19: 100%|██████████| 88/88 [00:05<00:00, 17.55it/s, train_loss_step=0.165, train_loss_epoch=0.154] 

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 88/88 [00:05<00:00, 17.55it/s, train_loss_step=0.165, train_loss_epoch=0.154]


#### · Validation

In [27]:
mpnn_valid = trainer_mpnn.validate(mpnn, val_loader)

/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Validation DataLoader 0: 100%|██████████| 25/25 [00:00<00:00, 28.57it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val/roc          │    0.8747327327728271     │
│         val_loss          │    0.18524500727653503    │
└───────────────────────────┴───────────────────────────┘

In [38]:
import torch

pred_probs = trainer_mpnn.predict(mpnn, dataloaders=val_loader)[:-1]
probs = torch.cat(pred_probs, dim=0).cpu().numpy()

all_y = []
for batch in val_loader:
    y = batch.Y  # 👈 get the label tensor from TrainingBatch
    all_y.append(y)
y_true = torch.cat(all_y, dim=0).cpu().numpy()  # shape: (num_samples, num_tasks)

model0_report_val = compute_classification_report(y_true, probs, threshold=0.6)
model0_report_val


/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 26/26 [00:00<00:00, 42.06it/s]
 - Precision: 0.7525
 - Recall: 0.2231
 - F1: 0.3195
 - AUC: 0.8455


,task,precision,recall,f1,auc,positives,predicted_positives,total_num
0,task_0,0.906250,0.483333,0.630435,0.805020,60,32,1501
1,task_1,0.862069,0.416667,0.561798,0.860970,60,29,1400
2,task_2,0.790123,0.395062,0.526749,0.911235,162,81,1349
3,task_3,0.000000,0.000000,0.000000,0.866571,56,0,1200
4,task_4,0.885714,0.182353,0.302439,0.729883,170,35,1269
5,task_5,0.800000,0.200000,0.320000,0.830411,80,20,1429
6,task_6,1.000000,0.075000,0.139535,0.857085,40,3,1342
7,task_7,0.704225,0.263158,0.383142,0.841941,190,71,1184
8,task_8,1.000000,0.037037,0.071429,0.850322,54,2,1470
9,task_9,0.733333,0.139241,0.234043,0.825183,79,15,1309


#### · Test

In [31]:
trainer_mpnn.test(mpnn, test_loader)

/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 13/13 [00:00<00:00, 29.29it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/roc          │    0.8697248697280884     │
└───────────────────────────┴───────────────────────────┘

[{'test/roc': 0.8697248697280884}]

In [37]:
pred_probs = trainer_mpnn.predict(mpnn, dataloaders=test_loader)
probs = torch.cat(pred_probs, dim=0).cpu().numpy()
all_y = []
for batch in test_loader:
    y = batch.Y  # 👈 get the label tensor from TrainingBatch
    all_y.append(y)
y_true = torch.cat(all_y, dim=0).cpu().numpy()  # shape: (num_samples, num_tasks)

model0_report_test = compute_classification_report(y_true, probs, threshold=0.5)
model0_report_test


/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 13/13 [00:00<00:00, 40.03it/s]
 - Precision: 0.6844
 - Recall: 0.2598
 - F1: 0.3518
 - AUC: 0.8388


,task,precision,recall,f1,auc,positives,predicted_positives,total_num
0,task_0,0.846154,0.343750,0.488889,0.709847,32,13,734
1,task_1,0.615385,0.444444,0.516129,0.854167,18,13,682
2,task_2,0.680000,0.441558,0.535433,0.925424,77,50,667
3,task_3,0.750000,0.083333,0.150000,0.837133,36,4,577
4,task_4,0.695652,0.197531,0.307692,0.777593,81,23,615
5,task_5,0.833333,0.294118,0.434783,0.864230,34,12,701
6,task_6,1.000000,0.117647,0.210526,0.831919,17,2,634
7,task_7,0.577778,0.282609,0.379562,0.826130,92,45,599
8,task_8,0.500000,0.064516,0.114286,0.826083,31,4,708
9,task_9,0.555556,0.142857,0.227273,0.863788,35,9,667


### 2) D-MPNN with Focal Loss

In [46]:
import torch.nn as nn


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        # self.alpha = alpha
        self.register_buffer("alpha", alpha)
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        # inputs, targets: (batch_size, num_tasks)
        mask = ~torch.isnan(targets)
        targets = targets.float().clone()
        targets[~mask] = 0  # avoid NaNs in loss

        probs = torch.sigmoid(inputs)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_factor = (1 - p_t) ** self.gamma
        alpha_factor = self.alpha * targets + (1 - self.alpha) * (1 - targets)

        bce_loss = self.bce(inputs, targets)
        loss = alpha_factor * focal_factor * bce_loss

        loss = loss * mask.float()
        return loss.sum() / mask.float().sum()
    
    

In [47]:
from sklearn.metrics import f1_score
import numpy as np

def compute_best_thresholds(y_true, y_probs, steps=20):
    thresholds = np.linspace(0.1, 0.9, steps)
    best_thresholds = []
    for i in range(y_true.shape[1]):
        mask = ~np.isnan(y_true[:, i])
        y_t = y_true[mask, i]
        y_p = y_probs[mask, i]

        best_f1 = 0
        best_thresh = 0.5
        for t in thresholds:
            pred = (y_p >= t).astype(int)
            f1 = f1_score(y_t, pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = t
        best_thresholds.append(best_thresh)
    print(f"Best thresholds: {best_thresholds}")
    return best_thresholds

In [ ]:
# import torch.nn as nn
# import torch.nn.functional as F
# import pytorch_lightning as pl
# from sklearn.metrics import recall_score, roc_auc_score
# import numpy as np




# class MPNNModel_FocalLoss(pl.LightningModule):

#     def __init__(self, mp, agg, ffn, batch_norm, metric_list, alpha_tensor):
#         super().__init__()
#         self.mp = mp
#         self.agg = agg
#         self.ffn = ffn
#         self.batch_norm = batch_norm
#         self.metric_list = metric_list

#         # ✅ Register alpha for automatic device transfer
#         self.loss_fn = FocalLoss(alpha=alpha_tensor, gamma=2.5)



#     def configure_optimizers(self):
#         return torch.optim.Adam(self.parameters(), lr=1e-3)
    
    
    
#     def forward(self, batch):
#         x = self.mp(batch.bmg)
#         x = self.agg(x, batch.bmg.batch)
#         x = self.ffn(x)
#         return x


#     def training_step(self, batch):
#         logits = self(batch)
#         targets = batch.Y
#         targets = torch.nan_to_num(targets, nan=0.0)
        
#         loss = self.loss_fn(logits, targets)
#         self.log("train_loss", loss)
#         return loss
    
    
            
#     def on_validation_epoch_start(self):
#         self.validation_outputs = []
        
#     def validation_step(self, batch):
#         logits = self(batch)
#         probs = torch.sigmoid(logits)
#         targets = batch.Y
#         self.validation_outputs.append({"probs": probs.detach(), "targets": targets.detach()})
        

#     def on_validation_epoch_end(self):

#         probs = torch.cat([x["probs"] for x in self.validation_outputs], dim=0).cpu().numpy()
#         targets = torch.cat([x["targets"] for x in self.validation_outputs], dim=0).cpu().numpy()

#         aucs = []
#         best_thresholds = compute_best_thresholds(targets, probs)
#         for i in range(targets.shape[1]):
#             mask = ~np.isnan(targets[:, i])
#             y_t = targets[mask, i]
#             y_p = probs[mask, i]
#             # pred = (y_p >= best_thresholds[i]).astype(int)

#             try:
#                 auc = roc_auc_score(y_t, y_p)
#                 aucs.append(auc)
#             except ValueError:
#                 continue  # skip task if not computable
        
#         # 🔹 Log overall mean AUC
#         if len(aucs) > 0:
#             mean_auc = np.mean(aucs)
#             self.log("val_mean_auc", mean_auc, prog_bar=True, on_epoch=True)

#         # self.log("Best thresholds", best_thresholds, prog_bar=True, on_epoch=True)
#         print('Best thresholds:', best_thresholds)
                

#     def on_test_epoch_start(self):
#         self.test_outputs = []

#     def test_step(self, batch, batch_idx):
#         logits = self(batch)
#         probs = torch.sigmoid(logits)
#         targets = batch.Y
#         self.test_outputs.append({"probs": probs.detach(), "targets": targets.detach()})

#     def on_test_epoch_end(self):
#         # Concatenate all test batches
#         probs = torch.cat([x["probs"] for x in self.test_outputs], dim=0).cpu().numpy()
#         targets = torch.cat([x["targets"] for x in self.test_outputs], dim=0).cpu().numpy()
#         n_tasks = targets.shape[1]

#         aucs = []
#         for i in range(n_tasks):
#             mask = ~np.isnan(targets[:, i])
#             y_true = targets[mask, i]
#             y_prob = probs[mask, i]
#             y_pred = (y_prob >= 0.5).astype(int)

#             # # 🔹 Log per-task recall
#             # if len(np.unique(y_true)) <= 1:
#             #     self.log(f"test_recall_task_{i}", 0.0, prog_bar=False, on_epoch=True)
#             # else:
#             #     recall = recall_score(y_true, y_pred, zero_division=0)
#             #     self.log(f"test_recall_task_{i}", recall, prog_bar=False, on_epoch=True)

#             # 🔹 AUC
#             try:
#                 auc = roc_auc_score(y_true, y_prob)
#                 aucs.append(auc)
#             except ValueError:
#                 continue  # skip task if AUC cannot be computed

#         # 🔹 Log overall mean AUC
#         if len(aucs) > 0:
#             mean_auc = np.mean(aucs)
#             self.log("test_mean_auc", mean_auc, prog_bar=True, on_epoch=True)

#         self.test_outputs.clear()
        
        
        
#     def predict_step(self, batch, batch_idx):
#         logits = self(batch)
#         probs = torch.sigmoid(logits)
#         return {"probs": probs.detach(), "targets": batch.Y.detach()}
    
    

In [119]:
from sklearn.metrics import recall_score, roc_auc_score
from torchmetrics.classification import MultilabelAUROC


class MPNNModel_FocalLoss(pl.LightningModule):


    def __init__(self, mp, agg, ffn, batch_norm, metric_list, alpha_tensor, gamma):
        super().__init__()
        self.mp = mp
        self.agg = agg
        self.ffn = ffn
        self.batch_norm = batch_norm
        self.metric_list = metric_list
        self.train_auc = MultilabelAUROC(num_labels=12, average='macro')  # for 12 binary tasks
        self.train_outputs = []

        # ✅ Register alpha for automatic device transfer
        self.loss_fn = FocalLoss(alpha=alpha_tensor, gamma=gamma)



    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)
    
    
    
    def forward(self, batch):
        x = self.mp(batch.bmg)
        x = self.agg(x, batch.bmg.batch)
        x = self.ffn(x)
        return x


    def training_step(self, batch, batch_idx):
        logits = self(batch)
        probs = torch.sigmoid(logits)
        targets = batch.Y
        mask = ~torch.isnan(targets)
        # task_mask = mask.float()
        targets = torch.nan_to_num(targets, nan=0.0)
        # loss = F.binary_cross_entropy_with_logits(
        #     logits, targets, weight=task_mask, pos_weight=self.pos_weight.to(logits.device)
        # )
        loss = self.loss_fn(logits, targets)
        self.log("train_loss", loss)
        
        # Save predictions/targets for AUC computation
        self.train_outputs.append({
            "probs": probs.detach(),
            "targets": targets.detach(),
            "mask": mask.detach()
        })
        return loss
    
    
    def on_train_epoch_end(self):
        probs = torch.cat([o["probs"] for o in self.train_outputs], dim=0)
        targets = torch.cat([o["targets"] for o in self.train_outputs], dim=0)
        mask = torch.cat([o["mask"] for o in self.train_outputs], dim=0)

        valid_rows = mask.any(dim=1)
        auc = self.train_auc(probs[valid_rows], targets[valid_rows].int())

        self.log("train_auc", auc, prog_bar=True)
        self.train_outputs.clear()  # clear for next epoch
    
    
            
    def on_validation_epoch_start(self):
        self.validation_outputs = []
        
    def validation_step(self, batch, batch_idx):
        logits = self(batch)
        probs = torch.sigmoid(logits)
        targets = batch.Y
        self.validation_outputs.append({"probs": probs.detach(), "targets": targets.detach()})
        

    def on_validation_epoch_end(self):
        
        probs = torch.cat([x["probs"] for x in self.validation_outputs], dim=0).cpu().numpy()
        targets = torch.cat([x["targets"] for x in self.validation_outputs], dim=0).cpu().numpy()

        aucs = []
        best_thresholds = compute_best_thresholds(targets, probs)
        for i in range(targets.shape[1]):
            mask = ~np.isnan(targets[:, i])
            y_t = targets[mask, i]
            y_p = probs[mask, i]
            # pred = (y_p >= best_thresholds[i]).astype(int)


            try:
                auc = roc_auc_score(y_t, y_p)
                aucs.append(auc)
            except ValueError:
                continue  # skip task if not computable
        
        # 🔹 Log overall mean AUC
        if len(aucs) > 0:
            mean_auc = np.mean(aucs)
            self.log("val_mean_auc", mean_auc, prog_bar=True, on_epoch=True)

        # self.log("Best thresholds", best_thresholds, prog_bar=True, on_epoch=True)
        print('Best thresholds:', best_thresholds)
                

    def on_test_epoch_start(self):
        self.test_outputs = []

    def test_step(self, batch, batch_idx):
        logits = self(batch)
        probs = torch.sigmoid(logits)
        targets = batch.Y
        self.test_outputs.append({"probs": probs.detach(), "targets": targets.detach()})

    def on_test_epoch_end(self):
        # Concatenate all test batches
        probs = torch.cat([x["probs"] for x in self.test_outputs], dim=0).cpu().numpy()
        targets = torch.cat([x["targets"] for x in self.test_outputs], dim=0).cpu().numpy()
        n_tasks = targets.shape[1]

        aucs = []
        for i in range(n_tasks):
            mask = ~np.isnan(targets[:, i])
            y_true = targets[mask, i]
            y_prob = probs[mask, i]
            y_pred = (y_prob >= 0.5).astype(int)

            # 🔹 Log per-task recall
            if len(np.unique(y_true)) <= 1:
                self.log(f"test_recall_task_{i}", 0.0, prog_bar=False, on_epoch=True)
            else:
                recall = recall_score(y_true, y_pred, zero_division=0)
                self.log(f"test_recall_task_{i}", recall, prog_bar=False, on_epoch=True)

            # 🔹 AUC
            try:
                auc = roc_auc_score(y_true, y_prob)
                aucs.append(auc)
            except ValueError:
                continue  # skip task if AUC cannot be computed

        # 🔹 Log overall mean AUC
        if len(aucs) > 0:
            mean_auc = np.mean(aucs)
            self.log("test_mean_auc", mean_auc, prog_bar=True, on_epoch=True)

        self.test_outputs.clear()
        
        
        
    def predict_step(self, batch, batch_idx):
        logits = self(batch)
        probs = torch.sigmoid(logits)
        return {"probs": probs.detach(), "targets": batch.Y.detach()}
    
    

In [120]:
alphas = []
for task in tox21_tasks:
    pos = tox21_data[task].sum()
    total = tox21_data[task].notna().sum()
    alpha = 1 - (pos / total)
    alphas.append(alpha)

# Create a tensor
alpha_tensor = torch.tensor(alphas, dtype=torch.float32)

In [121]:
import chemprop

mp = chemprop.nn.BondMessagePassing()
agg = chemprop.nn.MeanAggregation()

ffn = nn.Sequential(
    nn.Linear(300, 300),
    nn.BatchNorm1d(300),
    nn.ReLU(),
    nn.Dropout(0.7),
    nn.Linear(300, 12)
    )

# ffn = chemprop.nn.BinaryClassificationFFN(n_tasks=len(tox21_tasks), dropout=0.7)
        
batch_norm = False
mpnn_focal = MPNNModel_FocalLoss(mp, agg, ffn, batch_norm, metric_list, alpha_tensor, gamma=2.5)

#### · Train

In [122]:
trainer_focal_mpnn = pl.Trainer(max_epochs=20)
trainer_focal_mpnn.fit(mpnn_focal, train_loader, val_loader)

You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | mp        | BondMessagePassing | 227 K  | train
1 | agg       | MeanAggregation    | 0      | train
2 | ffn       | Sequential         | 94.5 K | train
3 | train_auc | MultilabelAUROC    | 0      | train
4 | loss_fn   | FocalLoss          | 0      | train
---------------------------------------------------------
322 K     Trainable params
0         Non-trainable params
322 K     Total params
1.289     Total estimated model params size (MB)
18        Modules in train mode
0         Modules in eval mode


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:02<00:00,  0.90it/s]Best thresholds: [np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.5210526315789473), np.float64(0.1), np.float64(0.4789473684210527)]
Best thresholds: [np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.5210526315789473), np.float64(0.1), np.float64(0.4789473684210527)]
                                                                           

/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 88/88 [01:21<00:00,  1.08it/s, v_num=16]Best thresholds: [np.float64(0.5210526315789473), np.float64(0.5210526315789473), np.float64(0.5210526315789473), np.float64(0.39473684210526316), np.float64(0.5210526315789473), np.float64(0.6052631578947368), np.float64(0.6473684210526316), np.float64(0.4368421052631579), np.float64(0.5631578947368421), np.float64(0.4368421052631579), np.float64(0.39473684210526316), np.float64(0.6052631578947368)]
Best thresholds: [np.float64(0.5210526315789473), np.float64(0.5210526315789473), np.float64(0.5210526315789473), np.float64(0.39473684210526316), np.float64(0.5210526315789473), np.float64(0.6052631578947368), np.float64(0.6473684210526316), np.float64(0.4368421052631579), np.float64(0.5631578947368421), np.float64(0.4368421052631579), np.float64(0.39473684210526316), np.float64(0.6052631578947368)]
Epoch 1: 100%|██████████| 88/88 [01:18<00:00,  1.13it/s, v_num=16, val_mean_auc=0.697, train_auc=0.608]Best thresholds: [np.fl

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 88/88 [01:51<00:00,  0.79it/s, v_num=16, val_mean_auc=0.827, train_auc=0.839]


#### · Validation

In [123]:
trainer_focal_mpnn.validate(mpnn_focal, val_loader)

/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Validation DataLoader 0: 100%|██████████| 25/25 [00:05<00:00,  4.30it/s]Best thresholds: [np.float64(0.6473684210526316), np.float64(0.6473684210526316), np.float64(0.5210526315789473), np.float64(0.4789473684210527), np.float64(0.4789473684210527), np.float64(0.6052631578947368), np.float64(0.5631578947368421), np.float64(0.4789473684210527), np.float64(0.6052631578947368), np.float64(0.6052631578947368), np.float64(0.5210526315789473), np.float64(0.5210526315789473)]
Best thresholds: [np.float64(0.6473684210526316), np.float64(0.6473684210526316), np.float64(0.5210526315789473), np.float64(0.4789473684210527), np.float64(0.4789473684210527), np.float64(0.6052631578947368), np.float64(0.5631578947368421), np.float64(0.4789473684210527), np.float64(0.6052631578947368), np.float64(0.6052631578947368), np.float64(0.5210526315789473), np.float64(0.5210526315789473)]
Validation DataLoader 0: 100%|██████████| 25/25 [00:05<00:00,  4.20it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       val_mean_auc        │    0.8272321224212646     │
└───────────────────────────┴───────────────────────────┘

[{'val_mean_auc': 0.8272321224212646}]

In [127]:
# preds = trainer_focal_mpnn.predict(mpnn_focal, dataloaders=val_loader)[:-1]

# probs = torch.cat([r["probs"] for r in preds], dim=0).cpu().numpy()
# y_true = torch.cat([r["targets"] for r in preds], dim=0).cpu().numpy()

model1_report_val = compute_classification_report(y_true, probs, threshold=0.55)
model1_report_val


 - Precision: 0.3918
 - Recall: 0.4523
 - F1: 0.3901
 - AUC: 0.8272


,task,precision,recall,f1,auc,positives,predicted_positives,total_num
0,task_0,0.283186,0.533333,0.369942,0.822213,60,113,1501
1,task_1,0.523810,0.550000,0.536585,0.840012,60,63,1400
2,task_2,0.638298,0.555556,0.594059,0.899022,162,141,1349
3,task_3,0.333333,0.142857,0.200000,0.843812,56,24,1200
4,task_4,0.595506,0.311765,0.409266,0.733977,170,89,1269
5,task_5,0.306122,0.562500,0.396476,0.796071,80,147,1429
6,task_6,0.243243,0.450000,0.315789,0.822043,40,74,1342
7,task_7,0.524476,0.394737,0.450450,0.820518,190,143,1184
8,task_8,0.162304,0.574074,0.253061,0.847575,54,191,1470
9,task_9,0.245033,0.468354,0.321739,0.800422,79,151,1309


#### · Test